# Build PBMC reference dataset for discovery cohort

In [1]:
suppressPackageStartupMessages({
  library(Seurat)
  library(future)
  library(tidyverse)
})

In [2]:
options(future.globals.maxSize = 1000000 * 1024^2, hpc.ncpus = 16)
plan(multicore, workers = getOption("hpc.ncpus", 1))

In [ ]:
results.dir <- "../results/tables/cite_seq/cohort_treatment_naive/cluster_annotate"
ref.dir <- "../data/ref/datasets"
dir.create(ref.dir, recursive = TRUE)

# Load data

### Full dataset

QS object generated by running Snakemake workflows in `01_prepare_internal_datasets`

In [3]:
seu <- qs::qread(file = "../data/processed/cite_seq/cohort_treatment_naive/cellbender/multi/peak-method:None_b:False_d:False/n-features:5000_normalisation:LogNormalize_clr:seurat_M:True_C:True/batch:orig.ident_normalisation:LogNormalize_integration:harmony/integrated.qs", nthreads = getOption("hpc.ncpus", 1))

In [4]:
object.size(seu) %>% format(unit = "Gb")

[1] "28.4 Gb"

In [5]:
seu

An object of class Seurat 
26712 features across 279855 samples within 3 assays 
Active assay: RNA (26424 features, 4993 variable features)
 3 layers present: data, counts, scale.data
 2 other assays present: ADT, ADTC
 4 dimensional reductions calculated: pca, integrated.rna, apca, integrated.adt

### Cluster annotation

In [ ]:
cluster.annotation <- read.table(
  file = file.path(results.dir, "cluster_annotation_final.tsv"),
  sep = "\t",
  header = TRUE,
  row.names = 1,
  stringsAsFactors = FALSE
) %>%
  mutate(
    cluster_fine = factor(
      cluster_fine,
      levels = c(
        "Artefact",
        "B_naive_transitional", "B_naive", "B_memory", "B_plasma",
        "DC_AXL_SIGLEC6", "DC_conventional_1", "DC_conventional_2", "DC_plasmacytoid",
        "Doublet", "Erythrocyte", "Granulocyte", "ILC",
        "Mono_CD14", "Mono_CD14_platelet", "Mono_CD14_IL1B", "Mono_CD14_IFN", "Mono_CD14_CD16", "Mono_CD16", "Mono_CD16_IFN",
        "NK_CD56bright", "NK_CD56dim",
        "Platelet", "Progenitor", "Proliferating",
        "T_CD4_naive", "T_CD4_naive_SOX4", "T_CD4_naive_IFN", "T_CD4_memory_central", "T_CD4_memory_central_IFN", "T_CD4_memory_effector",
        "T_regulatory_naive", "T_regulatory_memory",
        "T_CD8_naive", "T_CD8_naive_SOX4", "T_CD8_naive_IFN", "T_CD8_memory_central", "T_CD8_memory_effector",
        "T_MAIT", "T_GD", "T_DN"
      )
    ),
    cluster_coarse = factor(
      cluster_coarse,
      levels = c(
        "Artefact",
        "B_naive", "B_memory", "B_plasma",
        "DC_AXL_SIGLEC6", "DC_conventional", "DC_plasmacytoid",
        "Doublet", "Erythrocyte", "Granulocyte", "ILC",
        "Mono_CD14", "Mono_CD16",
        "NK_CD56bright", "NK_CD56dim",
        "Platelet", "Progenitor", "Proliferating",
        "T_CD4_naive", "T_CD4_memory",
        "T_regulatory_naive", "T_regulatory_memory",
        "T_CD8_naive", "T_CD8_memory",
        "T_MAIT", "T_GD", "T_DN"
      )
    ),
    cluster_main = factor(
      cluster_main,
      levels = c(
        "Artefact", "B", "DC", "Doublet", "Erythrocyte", "Granulocyte", "ILC", "Mono", "NK", "Platelet", "Progenitor", "Proliferating", "T"
      )
    )
  )
seu <- AddMetaData(seu, metadata = cluster.annotation)
Idents(seu) <- "cluster_coarse"

### Filtered dataset

In [7]:
# Remove 'Artefact' and 'Doublet' clusters as these are not real cell types/states
seu <- subset(seu, subset = cluster_main %in% c("Artefact", "Doublet"), invert = TRUE)
seu[[]] <- droplevels(seu[[]])

# Compute WNN graph and SPCA

In [8]:
seu <- seu %>%
  FindMultiModalNeighbors(
    reduction.list = list("integrated.rna", "integrated.adt"),
    dims.list = list(1:50, 1:50),
    modality.weight.name = c("RNA.weight", "ADT.weight"),
    verbose = TRUE
  ) %>%
  RunSPCA(
    assay = "RNA",
    graph = "wsnn",
    verbose = TRUE
  )

Calculating cell-specific modality weights

Finding 20 nearest neighbors for each modality.

Calculating kernel bandwidths

Finding multimodal neighbors

Constructing multimodal KNN graph

Constructing multimodal SNN graph

Computing sPCA transformation



# Compute UMAP embeddings

In [9]:
seu <- seu %>%
  RunUMAP(
    reduction = "integrated.rna",
    dims = 1:50,
    n.neighbors = 30L,
    metric = "cosine",
    min.dist = 0.3,
    spread = 1,
    n.epochs = 200,
    reduction.name = "umap",
    return.model = TRUE
  ) %>%
  RunUMAP(
    reduction = "integrated.adt",
    dims = 1:50,
    n.neighbors = 30L,
    metric = "cosine",
    min.dist = 0.3,
    spread = 1,
    n.epochs = 200,
    reduction.name = "aumap",
    return.model = TRUE
  ) %>% 
  RunUMAP(
    nn.name = "weighted.nn",
    n.neighbors = 30L,
    metric = "cosine",
    min.dist = 0.3,
    spread = 1,
    n.epochs = 200,
    reduction.name = "wnn.umap",
    return.model = TRUE
  )

Warning message:
"The default method for RunUMAP has changed from calling Python UMAP via reticulate to the R-native UWOT using the cosine metric
To use Python UMAP via reticulate, set umap.method to 'umap-learn' and metric to 'correlation'
This message will be shown once per session"
UMAP will return its model

19:55:52 UMAP embedding parameters a = 0.9922 b = 1.112

19:55:52 Read 265949 rows and found 50 numeric columns

19:55:52 Using Annoy for neighbor search, n_neighbors = 30

19:55:52 Building Annoy index with metric = cosine, n_trees = 50

0%   10   20   30   40   50   60   70   80   90   100%

[----|----|----|----|----|----|----|----|----|----|

*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
|

19:56:25 Writing NN index file to temp file /tmp/RtmpfT0ZkC/file39d87a5e2b6b19

19:56:26 Searching Annoy index using 16 threads, search_k = 3000

19:56:39 Annoy recall = 100%

19:56:46 Commencing smooth kNN distance calibration using 1

# Save reference

In [10]:
# Keep only necessary data to reduce object size
seu <- DietSeurat(
  seu,
  assays = c("RNA", "ADT"),
  dimreducs = c("pca", "apca", "spca", "umap", "aumap", "wnn.umap")
)

In [11]:
seu

An object of class Seurat 
26584 features across 265949 samples within 2 assays 
Active assay: RNA (26424 features, 4993 variable features)
 3 layers present: data, counts, scale.data
 1 other assay present: ADT
 6 dimensional reductions calculated: pca, apca, spca, umap, aumap, wnn.umap

In [ ]:
qs::qsave(seu, file = file.path(ref.dir, "pbmc_ocrelizumab_cohort_treatment_naive.qs"), nthreads = getOption("hpc.ncpus", 1))

# Session Info

In [14]:
sessionInfo()

R version 4.3.3 (2024-02-29)
Platform: x86_64-conda-linux-gnu (64-bit)
Running under: Ubuntu 22.04.5 LTS

Matrix products: default
BLAS/LAPACK: /ceph/project/fuggerlab/rfarooq/.conda/envs/sandbox/lib/libopenblasp-r0.3.28.so;  LAPACK version 3.12.0

locale:
[1] C

time zone: Europe/London
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
 [1] lubridate_1.9.3    forcats_1.0.0      stringr_1.5.1      dplyr_1.1.4       
 [5] purrr_1.0.2        readr_2.1.5        tidyr_1.3.1        tibble_3.2.1      
 [9] ggplot2_3.5.1      tidyverse_2.0.0    future_1.34.0      Seurat_5.1.0      
[13] SeuratObject_5.0.2 sp_2.1-4          

loaded via a namespace (and not attached):
  [1] RColorBrewer_1.1-3     jsonlite_1.8.9         magrittr_2.0.3        
  [4] spatstat.utils_3.1-0   farver_2.1.2           vctrs_0.6.5           
  [7] ROCR_1.0-11            spatstat.explore_3.2-6 base64enc_0.1-3       
 